# Mini-Lab C: TFRecord Pipeline & Benchmarking

**Goal:** Understand the TFRecord data format, build efficient tf.data pipelines, and benchmark CSV vs TFRecord performance for ML training.

**Dataset:** Chicago Taxi Trips (~750k rows from BigQuery public data)  
**Task:** Binary classification — predict whether a trip receives a tip (tip > 0)  
**Environment:** Local (no Vertex AI)

**What you'll learn:**
- How tf.Example and tf.train.Feature encode data into protocol buffers
- The three Feature types: FloatList, Int64List, BytesList
- Why TFRecord outperforms CSV for training pipelines
- How sharding enables parallel reads and better shuffling
- tf.data pipeline patterns: interleave, prefetch, batch
- When to choose TFRecord vs CSV vs other formats (exam-relevant)

**Exam relevance:** Data pipeline efficiency, TFRecord format, tf.data API, sharding strategy, training performance optimization.

---
## Section 1: Setup & Data Preparation

We'll query the Chicago Taxi Trips dataset from BigQuery, select relevant features, clean the data, and save a CSV baseline for benchmarking later.

In [1]:
# Imports
import os
import time
import glob
import struct
import hashlib
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from google.cloud import bigquery

warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

/Users/james.carty/Documents/VScode/google-ml-engineer/.venv-tf/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow version: 2.15.1
NumPy version: 1.26.4
Pandas version: 2.3.3


### 1.1 Constants

In [2]:
# Project config
PROJECT_ID = "carty-470812"
REGION = "us-central1"

# Local paths
DATA_DIR = "lab_c_data"
CSV_DIR = os.path.join(DATA_DIR, "csv")
TFRECORD_DIR = os.path.join(DATA_DIR, "tfrecord")
TFRECORD_SHARDED_DIR = os.path.join(DATA_DIR, "tfrecord_sharded")

for d in [DATA_DIR, CSV_DIR, TFRECORD_DIR, TFRECORD_SHARDED_DIR]:
    os.makedirs(d, exist_ok=True)

# Dataset config
SAMPLE_SIZE = 750_000
RANDOM_SEED = 42
TEST_FRACTION = 0.2

print("Directories created.")

Directories created.


### 1.2 Query Chicago Taxi Data from BigQuery

The Chicago Taxi Trips dataset has ~200M+ rows. We'll sample 750k and select features that give us a good mix of numeric, categorical, and string types — important for exercising all three tf.Feature types.

**Target:** `tip > 0` (binary classification — did the passenger tip?)

We filter for credit card payments since cash tips aren't reliably recorded, and apply basic quality filters.

In [3]:
client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT
    trip_seconds,
    trip_miles,
    fare,
    pickup_community_area,
    dropoff_community_area,
    payment_type,
    company,
    EXTRACT(HOUR FROM trip_start_timestamp) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM trip_start_timestamp) AS pickup_dayofweek,
    CASE WHEN tips > 0 THEN 1 ELSE 0 END AS tipped
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
WHERE
    fare > 0
    AND fare < 500
    AND trip_seconds > 0
    AND trip_seconds < 36000
    AND trip_miles > 0
    AND trip_miles < 500
    AND payment_type = 'Credit Card'
    AND trip_start_timestamp >= '2020-01-01'
ORDER BY RAND()
LIMIT {sample_size}
""".format(sample_size=SAMPLE_SIZE)

print(f"Querying {SAMPLE_SIZE:,} rows from Chicago Taxi Trips...")
df = client.query(query).to_dataframe()
print(f"Retrieved {len(df):,} rows, {len(df.columns)} columns")
df.head()

Querying 750,000 rows from Chicago Taxi Trips...
Retrieved 750,000 rows, 10 columns


,trip_seconds,trip_miles,fare,pickup_community_area,dropoff_community_area,payment_type,company,pickup_hour,pickup_dayofweek,tipped
0,753,0.97,8.00,<NA>,<NA>,Credit Card,Flash Cab,18,3,1
1,533,2.09,8.75,32,8,Credit Card,Flash Cab,17,1,1
2,780,2.80,10.50,33,32,Credit Card,Taxi Affiliation Services,16,3,1
3,1518,16.34,41.25,76,24,Credit Card,City Service,20,2,1
4,1140,8.00,23.00,32,77,Credit Card,Globe Taxi,16,6,1


### 1.3 Data Exploration & Cleaning

In [4]:
# Check the data
print("=== Shape ===")
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns\n")

print("=== Data Types ===")
print(df.dtypes)
print()

print("=== Null Counts ===")
print(df.isnull().sum())
print()

print("=== Target Distribution ===")
print(df['tipped'].value_counts(normalize=True).round(4))
print()

print("=== Numeric Summary ===")
df[['trip_seconds', 'trip_miles', 'fare', 'pickup_hour', 'pickup_dayofweek']].describe().round(2)

=== Shape ===
750,000 rows x 10 columns

=== Data Types ===
trip_seconds                Int64
trip_miles                float64
fare                      float64
pickup_community_area       Int64
dropoff_community_area      Int64
payment_type               object
company                    object
pickup_hour                 Int64
pickup_dayofweek            Int64
tipped                      Int64
dtype: object

=== Null Counts ===
trip_seconds                  0
trip_miles                    0
fare                          0
pickup_community_area     57238
dropoff_community_area    91367
payment_type                  0
company                       0
pickup_hour                   0
pickup_dayofweek              0
tipped                        0
dtype: int64

=== Target Distribution ===
tipped
1    0.9435
0    0.0565
Name: proportion, dtype: Float64

=== Numeric Summary ===


,trip_seconds,trip_miles,fare,pickup_hour,pickup_dayofweek
count,750000.0,750000.00,750000.00,750000.0,750000.0
mean,1324.41,8.14,24.64,14.2,4.01
std,1001.56,7.92,18.08,5.32,1.87
min,1.0,0.01,0.01,0.0,1.0
25%,540.0,1.28,8.00,11.0,2.0
50%,1072.0,3.98,18.00,15.0,4.0
75%,1894.0,15.90,42.00,18.0,6.0
max,34213.0,361.00,400.00,23.0,7.0


In [5]:
# Clean: fill nulls and standardize types
# Community areas can be null — fill with 0 (unknown)
df['pickup_community_area'] = df['pickup_community_area'].fillna(0).astype(int)
df['dropoff_community_area'] = df['dropoff_community_area'].fillna(0).astype(int)

# Company can be null — fill with "Unknown"
df['company'] = df['company'].fillna('Unknown')

# Payment type is always 'Credit Card' (we filtered) — drop it since it's constant
df = df.drop(columns=['payment_type'])

# Verify no nulls remain
assert df.isnull().sum().sum() == 0, "Still have nulls!"
print(f"Clean dataset: {len(df):,} rows, {len(df.columns)} columns")
print(f"Null count: {df.isnull().sum().sum()}")
print(f"\nFinal columns: {list(df.columns)}")

Clean dataset: 750,000 rows, 9 columns
Null count: 0

Final columns: ['trip_seconds', 'trip_miles', 'fare', 'pickup_community_area', 'dropoff_community_area', 'company', 'pickup_hour', 'pickup_dayofweek', 'tipped']


### 1.4 Train/Test Split & Save CSV Baseline

We use a hash-based split for reproducibility — same approach as earlier labs.

In [6]:
# Hash-based split for reproducibility
def hash_split(df, test_fraction, seed=42):
    """Deterministic split using row hash."""
    hash_values = df.apply(
        lambda row: int(hashlib.md5(str(row.values).encode()).hexdigest(), 16),
        axis=1
    )
    is_test = (hash_values % 100) < (test_fraction * 100)
    return df[~is_test].copy(), df[is_test].copy()

df_train, df_test = hash_split(df, TEST_FRACTION, RANDOM_SEED)

print(f"Train: {len(df_train):,} rows ({len(df_train)/len(df)*100:.1f}%)")
print(f"Test:  {len(df_test):,} rows ({len(df_test)/len(df)*100:.1f}%)")
print(f"\nTarget distribution (train): {df_train['tipped'].mean():.4f}")
print(f"Target distribution (test):  {df_test['tipped'].mean():.4f}")

Train: 600,189 rows (80.0%)
Test:  149,811 rows (20.0%)

Target distribution (train): 0.9433
Target distribution (test):  0.9441


In [7]:
# Save CSV baseline for benchmarking
train_csv_path = os.path.join(CSV_DIR, "train.csv")
test_csv_path = os.path.join(CSV_DIR, "test.csv")

df_train.to_csv(train_csv_path, index=False)
df_test.to_csv(test_csv_path, index=False)

train_csv_size = os.path.getsize(train_csv_path) / (1024 * 1024)
test_csv_size = os.path.getsize(test_csv_path) / (1024 * 1024)

print(f"Train CSV: {train_csv_path} ({train_csv_size:.1f} MB)")
print(f"Test CSV:  {test_csv_path} ({test_csv_size:.1f} MB)")

Train CSV: lab_c_data/csv/train.csv (26.1 MB)
Test CSV:  lab_c_data/csv/test.csv (6.5 MB)


---
## Section 2: TFRecord Fundamentals

### What is TFRecord?

TFRecord is TensorFlow's native binary storage format. Under the hood, each record is a **serialized protocol buffer** (protobuf) — specifically a `tf.train.Example`.

**Why not just CSV?**
- CSV requires parsing every field from text on every read
- TFRecord is pre-serialized binary — just deserialize, no parsing
- TFRecord supports streaming reads (no need to load everything into memory)
- Sharded TFRecords enable parallel I/O across multiple files

### The tf.train.Example Schema

Every TFRecord contains `tf.train.Example` messages, which have a `Features` field containing a map of feature names to `Feature` values. Each `Feature` is one of three types:

| tf.train.Feature type | Python type | Use for |
|---|---|---|
| `FloatList` | `float` | Continuous numeric (fare, distance) |
| `Int64List` | `int` | Integers, encoded categoricals, labels |
| `BytesList` | `bytes` / `str` | Strings, raw bytes, variable-length data |

**Key exam concept:** You must define the schema (feature description) at both write time AND read time. There's no self-describing header like CSV — if you lose your schema, you can't read your TFRecords.

### 2.1 Helper Functions

These helpers convert Python/NumPy values into tf.train.Feature objects, then assemble them into a tf.train.Example.

In [8]:
# === Feature helper functions ===
# Each converts a value into the appropriate tf.train.Feature type

def _float_feature(value):
    """Create a FloatList Feature from a float/numpy scalar."""
    return tf.train.Feature(float_list=tf.train.FloatList(value=[float(value)]))

def _int64_feature(value):
    """Create an Int64List Feature from an int/numpy scalar."""
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(value)]))

def _bytes_feature(value):
    """Create a BytesList Feature from a string."""
    if isinstance(value, str):
        value = value.encode('utf-8')
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

# === Define our feature schema ===
# This maps column names to their Feature types — used for both writing and reading

NUMERIC_FEATURES = ['trip_seconds', 'trip_miles', 'fare']
INT_FEATURES = ['pickup_community_area', 'dropoff_community_area', 'pickup_hour', 'pickup_dayofweek']
STRING_FEATURES = ['company']
LABEL = 'tipped'

def create_example(row):
    """Convert a DataFrame row into a tf.train.Example."""
    feature = {}

    # Numeric features → FloatList
    for col in NUMERIC_FEATURES:
        feature[col] = _float_feature(row[col])

    # Integer features → Int64List
    for col in INT_FEATURES:
        feature[col] = _int64_feature(row[col])

    # String features → BytesList
    for col in STRING_FEATURES:
        feature[col] = _bytes_feature(row[col])

    # Label → Int64List
    feature[LABEL] = _int64_feature(row[LABEL])

    example = tf.train.Example(
        features=tf.train.Features(feature=feature)
    )
    return example

# Test it with one row
sample_row = df_train.iloc[0]
example = create_example(sample_row)
print("Sample tf.train.Example created successfully")
print(f"Serialized size: {len(example.SerializeToString())} bytes")
print(f"\nOriginal row:\n{sample_row}")
print(f"\nProtobuf features: {list(example.features.feature.keys())}")

Sample tf.train.Example created successfully
Serialized size: 228 bytes

Original row:
trip_seconds                    753
trip_miles                     0.97
fare                            8.0
pickup_community_area             0
dropoff_community_area            0
company                   Flash Cab
pickup_hour                      18
pickup_dayofweek                  3
tipped                            1
Name: 0, dtype: object

Protobuf features: ['company', 'pickup_hour', 'trip_seconds', 'pickup_community_area', 'trip_miles', 'pickup_dayofweek', 'dropoff_community_area', 'fare', 'tipped']


### 2.2 Write a Single TFRecord File

Now we serialize all training rows into one TFRecord file. This is the simplest case — no sharding yet.

In [9]:
def write_tfrecord(df, output_path):
    """Write a DataFrame to a single TFRecord file."""
    writer = tf.io.TFRecordWriter(output_path)
    count = 0
    for _, row in df.iterrows():
        example = create_example(row)
        writer.write(example.SerializeToString())
        count += 1
        if count % 100_000 == 0:
            print(f"  Written {count:,} records...")
    writer.close()
    return count

train_tfrecord_path = os.path.join(TFRECORD_DIR, "train.tfrecord")

print("Writing training data to single TFRecord file...")
start = time.time()
n_written = write_tfrecord(df_train, train_tfrecord_path)
write_time = time.time() - start

tfrecord_size = os.path.getsize(train_tfrecord_path) / (1024 * 1024)
print(f"\nDone! Wrote {n_written:,} records in {write_time:.1f}s")
print(f"File size: {tfrecord_size:.1f} MB")
print(f"CSV size:  {train_csv_size:.1f} MB")
print(f"Size ratio: TFRecord is {tfrecord_size/train_csv_size:.2f}x the CSV size")

Writing training data to single TFRecord file...
  Written 100,000 records...
  Written 200,000 records...
  Written 300,000 records...
  Written 400,000 records...
  Written 500,000 records...
  Written 600,000 records...

Done! Wrote 600,189 records in 29.7s
File size: 144.8 MB
CSV size:  26.1 MB
Size ratio: TFRecord is 5.54x the CSV size


### 2.3 Read TFRecords Back — The Feature Description

**This is the key concept:** To read TFRecords, you need a `feature_description` dict that tells TensorFlow what type and shape each feature has. Without this, the binary data is meaningless.

This is fundamentally different from CSV, where the header row is self-describing. With TFRecord, the schema lives in your code, not in the file.

In [10]:
# === Feature description for parsing ===
# This is the READ-TIME schema. Must match what we wrote.
# FixedLenFeature = every record has exactly this feature with this shape

feature_description = {}

# Numeric → float32 scalar
for col in NUMERIC_FEATURES:
    feature_description[col] = tf.io.FixedLenFeature([], tf.float32)

# Integer → int64 scalar
for col in INT_FEATURES:
    feature_description[col] = tf.io.FixedLenFeature([], tf.int64)

# String → bytes scalar
for col in STRING_FEATURES:
    feature_description[col] = tf.io.FixedLenFeature([], tf.string)

# Label → int64 scalar
feature_description[LABEL] = tf.io.FixedLenFeature([], tf.int64)

def parse_tfrecord(serialized_example):
    """Parse a single serialized tf.train.Example."""
    return tf.io.parse_single_example(serialized_example, feature_description)

# Test: read back and verify
raw_dataset = tf.data.TFRecordDataset(train_tfrecord_path)
parsed_dataset = raw_dataset.map(parse_tfrecord)

# Check first record
for record in parsed_dataset.take(1):
    print("=== Parsed TFRecord (first record) ===")
    for key, value in sorted(record.items()):
        print(f"  {key}: {value.numpy()}")

print(f"\n=== Original CSV row (first) ===")
for col in sorted(df_train.columns):
    print(f"  {col}: {df_train.iloc[0][col]}")

=== Parsed TFRecord (first record) ===
  company: b'Flash Cab'
  dropoff_community_area: 0
  fare: 8.0
  pickup_community_area: 0
  pickup_dayofweek: 3
  pickup_hour: 18
  tipped: 1
  trip_miles: 0.9700000286102295
  trip_seconds: 753.0

=== Original CSV row (first) ===
  company: Flash Cab
  dropoff_community_area: 0
  fare: 8.0
  pickup_community_area: 0
  pickup_dayofweek: 3
  pickup_hour: 18
  tipped: 1
  trip_miles: 0.97
  trip_seconds: 753


### 2.4 Verify Round-Trip Correctness

Let's make sure we haven't lost or corrupted any data by checking multiple records.

In [11]:
# Count records in TFRecord file
def count_tfrecords(path):
    count = 0
    for _ in tf.data.TFRecordDataset(path):
        count += 1
    return count

n_in_file = count_tfrecords(train_tfrecord_path)
print(f"Records in TFRecord file: {n_in_file:,}")
print(f"Rows in training DataFrame: {len(df_train):,}")
assert n_in_file == len(df_train), "Count mismatch!"
print("\nRow counts match.")

# Spot-check a few numeric values
print("\nSpot-checking values...")
for i, record in enumerate(parsed_dataset.take(5)):
    orig_fare = df_train.iloc[i]['fare']
    tfr_fare = record['fare'].numpy()
    match = np.isclose(orig_fare, tfr_fare, atol=1e-4)
    print(f"  Row {i}: CSV fare={orig_fare:.2f}, TFRecord fare={tfr_fare:.2f}, match={match}")

Records in TFRecord file: 600,189
Rows in training DataFrame: 600,189

Row counts match.

Spot-checking values...
  Row 0: CSV fare=8.00, TFRecord fare=8.00, match=True
  Row 1: CSV fare=8.75, TFRecord fare=8.75, match=True
  Row 2: CSV fare=10.50, TFRecord fare=10.50, match=True
  Row 3: CSV fare=41.25, TFRecord fare=41.25, match=True
  Row 4: CSV fare=7.25, TFRecord fare=7.25, match=True


---
## Section 3: Sharding

### Why Shard?

A single massive TFRecord file has two problems:
1. **Sequential bottleneck** — Only one reader can access it at a time
2. **Poor shuffling** — `tf.data.TFRecordDataset.shuffle()` uses an in-memory buffer; if the buffer is smaller than the dataset, records near each other in the file tend to stay near each other

Sharding solves both:
- **Parallel reads:** `tf.data.Dataset.interleave()` can read from multiple shards simultaneously
- **Better shuffling:** Shuffle the list of shard files, then shuffle within each shard's buffer — gives much better randomization

### How Many Shards?

**Rules of thumb:**
- At least as many shards as training workers (for distributed training)
- Each shard should be 100MB–200MB for efficient I/O
- More shards = better shuffling, but too many = filesystem overhead
- For our ~750k rows: 10 shards is a reasonable choice

In [12]:
NUM_SHARDS = 10

def write_sharded_tfrecords(df, output_dir, num_shards):
    """Write a DataFrame to multiple sharded TFRecord files."""
    # Shuffle the dataframe first to randomize shard contents
    df_shuffled = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    rows_per_shard = len(df_shuffled) // num_shards
    total_written = 0
    shard_sizes = []

    for shard_id in range(num_shards):
        shard_path = os.path.join(output_dir, f"train-{shard_id:05d}-of-{num_shards:05d}.tfrecord")
        start_idx = shard_id * rows_per_shard
        # Last shard gets remaining rows
        end_idx = len(df_shuffled) if shard_id == num_shards - 1 else (shard_id + 1) * rows_per_shard
        shard_df = df_shuffled.iloc[start_idx:end_idx]

        writer = tf.io.TFRecordWriter(shard_path)
        for _, row in shard_df.iterrows():
            example = create_example(row)
            writer.write(example.SerializeToString())
        writer.close()

        shard_size = os.path.getsize(shard_path) / (1024 * 1024)
        shard_sizes.append(shard_size)
        total_written += len(shard_df)
        print(f"  Shard {shard_id}: {len(shard_df):,} records ({shard_size:.1f} MB)")

    return total_written, shard_sizes

print(f"Writing {NUM_SHARDS} sharded TFRecord files...")
start = time.time()
n_sharded, shard_sizes = write_sharded_tfrecords(df_train, TFRECORD_SHARDED_DIR, NUM_SHARDS)
shard_time = time.time() - start

print(f"\nTotal records written: {n_sharded:,}")
print(f"Time: {shard_time:.1f}s")
print(f"Avg shard size: {np.mean(shard_sizes):.1f} MB")
print(f"Total sharded size: {sum(shard_sizes):.1f} MB")

Writing 10 sharded TFRecord files...
  Shard 0: 60,018 records (14.5 MB)
  Shard 1: 60,018 records (14.5 MB)
  Shard 2: 60,018 records (14.5 MB)
  Shard 3: 60,018 records (14.5 MB)
  Shard 4: 60,018 records (14.5 MB)
  Shard 5: 60,018 records (14.5 MB)
  Shard 6: 60,018 records (14.5 MB)
  Shard 7: 60,018 records (14.5 MB)
  Shard 8: 60,018 records (14.5 MB)
  Shard 9: 60,027 records (14.5 MB)

Total records written: 600,189
Time: 30.2s
Avg shard size: 14.5 MB
Total sharded size: 144.8 MB


In [13]:
# Verify shard counts add up
shard_files = sorted(glob.glob(os.path.join(TFRECORD_SHARDED_DIR, "*.tfrecord")))
print(f"Shard files found: {len(shard_files)}")

total_records = 0
for f in shard_files:
    n = count_tfrecords(f)
    total_records += n

print(f"Total records across shards: {total_records:,}")
print(f"Expected: {len(df_train):,}")
assert total_records == len(df_train), "Shard count mismatch!"
print("\nAll shard counts verified.")

Shard files found: 10
Total records across shards: 600,189
Expected: 600,189

All shard counts verified.


### File Size Comparison

In [14]:
# Compare file sizes
csv_total = train_csv_size
tfrecord_single = tfrecord_size
tfrecord_sharded_total = sum(shard_sizes)

print("=== File Size Comparison (Training Data) ===")
print(f"  CSV:              {csv_total:.1f} MB")
print(f"  TFRecord (single): {tfrecord_single:.1f} MB")
print(f"  TFRecord (sharded): {tfrecord_sharded_total:.1f} MB ({NUM_SHARDS} files)")
print()
print("Note: TFRecord files are typically similar in size or slightly larger")
print("than CSV because of protobuf overhead. The performance gain comes from")
print("read speed, not compression. (You can add GZIP compression to TFRecords")
print("for size reduction, but that adds CPU overhead at read time.)")

=== File Size Comparison (Training Data) ===
  CSV:              26.1 MB
  TFRecord (single): 144.8 MB
  TFRecord (sharded): 144.8 MB (10 files)

Note: TFRecord files are typically similar in size or slightly larger
than CSV because of protobuf overhead. The performance gain comes from
read speed, not compression. (You can add GZIP compression to TFRecords
for size reduction, but that adds CPU overhead at read time.)


---
## Section 4: tf.data Pipelines

Now we build three different input pipelines that produce identical (features, label) tuples, so we can benchmark them fairly.

### Pipeline Architecture

All three pipelines end with the same structure:
1. **Read** — load raw data from disk
2. **Parse** — convert to tensors
3. **Shuffle** — randomize order
4. **Batch** — group into training batches
5. **Prefetch** — overlap I/O with training compute

The difference is steps 1-2: how each format gets data from disk into tensors.

In [15]:
BATCH_SIZE = 512
SHUFFLE_BUFFER = 10_000

# === Define which features are which type for the model ===
# (We'll need this for all three pipelines)

ALL_FEATURE_COLS = NUMERIC_FEATURES + INT_FEATURES + STRING_FEATURES

def prepare_features_and_label(features_dict):
    """Convert parsed dict → (features_tensor, label) for model training.

    For simplicity, we:
    - Keep numeric features as-is
    - Cast int features to float32
    - Hash string features to integers, then cast to float32
    - Stack everything into a single feature vector
    """
    feature_tensors = []

    for col in NUMERIC_FEATURES:
        feature_tensors.append(tf.cast(features_dict[col], tf.float32))

    for col in INT_FEATURES:
        feature_tensors.append(tf.cast(features_dict[col], tf.float32))

    for col in STRING_FEATURES:
        # Simple hash encoding for strings → numeric
        hashed = tf.strings.to_hash_bucket_fast(features_dict[col], num_buckets=1000)
        feature_tensors.append(tf.cast(hashed, tf.float32))

    features = tf.stack(feature_tensors)
    label = tf.cast(features_dict[LABEL], tf.float32)
    return features, label

### 4.1 Pipeline A: CSV via tf.data

In [16]:
def build_csv_pipeline(csv_path, batch_size=BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER):
    """Build a tf.data pipeline that reads from CSV."""

    # Define column names and defaults (for tf.data CSV reader)
    column_names = NUMERIC_FEATURES + INT_FEATURES + STRING_FEATURES + [LABEL]

    # We need to know the column order in the CSV
    # Read header to get column order
    with open(csv_path, 'r') as f:
        header = f.readline().strip().split(',')

    column_defaults = []
    for col in header:
        if col in NUMERIC_FEATURES:
            column_defaults.append([0.0])
        elif col in INT_FEATURES or col == LABEL:
            column_defaults.append([0])
        elif col in STRING_FEATURES:
            column_defaults.append([''])
        else:
            column_defaults.append([0.0])  # fallback

    dataset = tf.data.experimental.make_csv_dataset(
        csv_path,
        batch_size=1,  # We'll rebatch later for fair comparison
        column_names=header,
        column_defaults=None,
        label_name=LABEL,
        num_epochs=1,
        shuffle=False,  # We'll shuffle ourselves
    )

    # make_csv_dataset returns (features_dict, label) — restructure to match our pattern
    def restructure(features_dict, label):
        features_dict[LABEL] = label
        return features_dict

    dataset = dataset.map(restructure)
    # Unbatch the batch_size=1 so we get individual records
    dataset = dataset.unbatch()
    dataset = dataset.map(prepare_features_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

# Test it
csv_ds = build_csv_pipeline(train_csv_path)
for features, labels in csv_ds.take(1):
    print(f"CSV pipeline — features shape: {features.shape}, labels shape: {labels.shape}")
    print(f"Sample features: {features[0].numpy()}")
    print(f"Sample label: {labels[0].numpy()}")

CSV pipeline — features shape: (512, 8), labels shape: (512,)
Sample features: [1380.     7.4   22.    32.     6.    19.     5.   959. ]
Sample label: 1.0


### 4.2 Pipeline B: Single TFRecord

In [17]:
def build_tfrecord_pipeline(tfrecord_path, batch_size=BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER):
    """Build a tf.data pipeline from a single TFRecord file."""
    dataset = tf.data.TFRecordDataset(tfrecord_path)
    dataset = dataset.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(prepare_features_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# Test it
tfr_ds = build_tfrecord_pipeline(train_tfrecord_path)
for features, labels in tfr_ds.take(1):
    print(f"TFRecord pipeline — features shape: {features.shape}, labels shape: {labels.shape}")
    print(f"Sample features: {features[0].numpy()}")
    print(f"Sample label: {labels[0].numpy()}")

TFRecord pipeline — features shape: (512, 8), labels shape: (512,)
Sample features: [2209.     17.86   44.25   76.      8.     18.      3.    218.  ]
Sample label: 0.0


### 4.3 Pipeline C: Sharded TFRecords with Interleave

This is the production pattern. `interleave()` reads from multiple shards simultaneously, which:
- Increases I/O throughput (parallel reads)
- Improves shuffle quality (mixing records from different shards)
- Mirrors what distributed training does automatically

In [18]:
def build_sharded_tfrecord_pipeline(shard_pattern, batch_size=BATCH_SIZE, shuffle_buffer=SHUFFLE_BUFFER):
    """Build a tf.data pipeline from sharded TFRecord files."""
    # List all shard files
    shard_files = tf.data.Dataset.list_files(shard_pattern, shuffle=True)

    # Interleave reads from multiple shards
    # cycle_length = how many shards to read from simultaneously
    # block_length = how many records to read from each shard before switching
    dataset = shard_files.interleave(
        lambda path: tf.data.TFRecordDataset(path),
        cycle_length=NUM_SHARDS,
        block_length=16,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=False  # Allow non-deterministic ordering for speed
    )

    dataset = dataset.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(prepare_features_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(shuffle_buffer)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

shard_pattern = os.path.join(TFRECORD_SHARDED_DIR, "*.tfrecord")

# Test it
sharded_ds = build_sharded_tfrecord_pipeline(shard_pattern)
for features, labels in sharded_ds.take(1):
    print(f"Sharded TFRecord pipeline — features shape: {features.shape}, labels shape: {labels.shape}")
    print(f"Sample features: {features[0].numpy()}")
    print(f"Sample label: {labels[0].numpy()}")

Sharded TFRecord pipeline — features shape: (512, 8), labels shape: (512,)
Sample features: [2287.    17.6   45.5   76.     0.    12.     5.   679. ]
Sample label: 1.0


---
## Section 5: Benchmark

Now the payoff — we time how fast each pipeline can deliver data. We'll iterate through the entire training set and measure throughput.

**What we're measuring:** End-to-end data delivery speed — from disk to batched tensors ready for training. This includes reading, parsing, shuffling, and batching.

We run multiple iterations to get stable numbers.

In [19]:
def benchmark_pipeline(pipeline_fn, name, n_runs=3):
    """Benchmark a pipeline by iterating through the full dataset multiple times."""
    times = []

    for run in range(n_runs):
        dataset = pipeline_fn()
        n_batches = 0
        n_records = 0

        start = time.time()
        for features, labels in dataset:
            n_batches += 1
            n_records += features.shape[0]
        elapsed = time.time() - start

        times.append(elapsed)
        if run == 0:
            print(f"  {name}: {n_records:,} records in {n_batches:,} batches")

    avg_time = np.mean(times)
    std_time = np.std(times)
    throughput = n_records / avg_time

    print(f"  {name}: {avg_time:.2f}s ± {std_time:.2f}s ({throughput:,.0f} records/sec)")
    return {
        'name': name,
        'avg_time': avg_time,
        'std_time': std_time,
        'throughput': throughput,
        'n_records': n_records
    }

print("=== Pipeline Benchmark ===\n")
print("Running CSV pipeline...")
csv_result = benchmark_pipeline(
    lambda: build_csv_pipeline(train_csv_path),
    "CSV (tf.data)"
)

print("\nRunning single TFRecord pipeline...")
tfr_result = benchmark_pipeline(
    lambda: build_tfrecord_pipeline(train_tfrecord_path),
    "TFRecord (single)"
)

print("\nRunning sharded TFRecord pipeline...")
sharded_result = benchmark_pipeline(
    lambda: build_sharded_tfrecord_pipeline(shard_pattern),
    "TFRecord (sharded)"
)

=== Pipeline Benchmark ===

Running CSV pipeline...
  CSV (tf.data): 600,189 records in 1,173 batches
  CSV (tf.data): 11.43s ± 0.18s (52,521 records/sec)

Running single TFRecord pipeline...
  TFRecord (single): 600,189 records in 1,173 batches
  TFRecord (single): 7.22s ± 0.01s (83,096 records/sec)

Running sharded TFRecord pipeline...
  TFRecord (sharded): 600,189 records in 1,173 batches
  TFRecord (sharded): 9.88s ± 0.11s (60,753 records/sec)


In [20]:
# Summary comparison
results = [csv_result, tfr_result, sharded_result]

print("\n" + "=" * 65)
print(f"{'Pipeline':<25} {'Time (s)':<15} {'Throughput':<20}")
print("=" * 65)
for r in results:
    print(f"{r['name']:<25} {r['avg_time']:.2f} ± {r['std_time']:.2f}    {r['throughput']:>12,.0f} rec/s")
print("=" * 65)

# Speedup calculations
csv_time = csv_result['avg_time']
for r in results[1:]:
    speedup = csv_time / r['avg_time']
    print(f"\n{r['name']} is {speedup:.1f}x {'faster' if speedup > 1 else 'slower'} than CSV")


Pipeline                  Time (s)        Throughput          
CSV (tf.data)             11.43 ± 0.18          52,521 rec/s
TFRecord (single)         7.22 ± 0.01          83,096 rec/s
TFRecord (sharded)        9.88 ± 0.11          60,753 rec/s

TFRecord (single) is 1.6x faster than CSV

TFRecord (sharded) is 1.2x faster than CSV


---
## Section 6: Train & Compare

Now let's train the same model on CSV vs sharded TFRecord pipelines and compare training time per epoch. The model is intentionally simple — the point is pipeline comparison, not model accuracy.

In [21]:
def build_model():
    """Simple dense model for binary classification."""
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(8,)),  # 8 features
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                576       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 16)                528       
                                                                 
 dense_3 (Dense)             (None, 1)                 17        
                                                                 
Total params: 3201 (12.50 KB)
Trainable params: 3201 (12.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### 6.1 Build Test Pipeline

We need a test dataset for evaluation. We'll use TFRecord format for test data too.

In [22]:
# Write test data to TFRecord
test_tfrecord_path = os.path.join(TFRECORD_DIR, "test.tfrecord")
print("Writing test data to TFRecord...")
n_test_written = write_tfrecord(df_test, test_tfrecord_path)
print(f"Wrote {n_test_written:,} test records")

def build_test_pipeline(path):
    """Build a non-shuffled test pipeline."""
    dataset = tf.data.TFRecordDataset(path)
    dataset = dataset.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(prepare_features_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

test_ds = build_test_pipeline(test_tfrecord_path)
print("Test pipeline ready.")

Writing test data to TFRecord...
  Written 100,000 records...
Wrote 149,811 test records
Test pipeline ready.


### 6.2 Train on CSV Pipeline

In [23]:
EPOCHS = 5

print("=== Training on CSV Pipeline ===\n")
model_csv = build_model()

csv_train_ds = build_csv_pipeline(train_csv_path)

start = time.time()
history_csv = model_csv.fit(
    csv_train_ds,
    epochs=EPOCHS,
    verbose=1
)
csv_train_time = time.time() - start

# Evaluate on test set
csv_test_results = model_csv.evaluate(test_ds, verbose=0)
print(f"\nCSV Training: {csv_train_time:.1f}s total ({csv_train_time/EPOCHS:.1f}s per epoch)")
print(f"CSV Test accuracy: {csv_test_results[1]:.4f}")

=== Training on CSV Pipeline ===

Epoch 1/5
1173/1173 [==============================] - 12s 10ms/step - loss: 0.6617 - accuracy: 0.9162
Epoch 2/5
1173/1173 [==============================] - 12s 10ms/step - loss: 0.4541 - accuracy: 0.9288
Epoch 3/5
1173/1173 [==============================] - 12s 10ms/step - loss: 0.3253 - accuracy: 0.9366
Epoch 4/5
1173/1173 [==============================] - 12s 10ms/step - loss: 0.2810 - accuracy: 0.9396
Epoch 5/5
1173/1173 [==============================] - 12s 10ms/step - loss: 0.2667 - accuracy: 0.9404

CSV Training: 58.3s total (11.7s per epoch)
CSV Test accuracy: 0.9441


### 6.3 Train on Sharded TFRecord Pipeline

In [24]:
print("=== Training on Sharded TFRecord Pipeline ===\n")
model_tfr = build_model()

sharded_train_ds = build_sharded_tfrecord_pipeline(shard_pattern)

start = time.time()
history_tfr = model_tfr.fit(
    sharded_train_ds,
    epochs=EPOCHS,
    verbose=1
)
tfr_train_time = time.time() - start

# Evaluate on same test set
tfr_test_results = model_tfr.evaluate(test_ds, verbose=0)
print(f"\nTFRecord Training: {tfr_train_time:.1f}s total ({tfr_train_time/EPOCHS:.1f}s per epoch)")
print(f"TFRecord Test accuracy: {tfr_test_results[1]:.4f}")

=== Training on Sharded TFRecord Pipeline ===

Epoch 1/5
1173/1173 [==============================] - 10s 9ms/step - loss: 0.2837 - accuracy: 0.9413
Epoch 2/5
1173/1173 [==============================] - 10s 8ms/step - loss: 0.2310 - accuracy: 0.9432
Epoch 3/5
1173/1173 [==============================] - 10s 9ms/step - loss: 0.2249 - accuracy: 0.9433
Epoch 4/5
1173/1173 [==============================] - 10s 9ms/step - loss: 0.2222 - accuracy: 0.9433
Epoch 5/5
1173/1173 [==============================] - 10s 9ms/step - loss: 0.2195 - accuracy: 0.9433

TFRecord Training: 51.6s total (10.3s per epoch)
TFRecord Test accuracy: 0.9441


### 6.4 Training Comparison

In [25]:
print("=" * 55)
print(f"{'Metric':<30} {'CSV':<12} {'TFRecord':<12}")
print("=" * 55)
print(f"{'Total training time (s)':<30} {csv_train_time:<12.1f} {tfr_train_time:<12.1f}")
print(f"{'Time per epoch (s)':<30} {csv_train_time/EPOCHS:<12.1f} {tfr_train_time/EPOCHS:<12.1f}")
print(f"{'Test accuracy':<30} {csv_test_results[1]:<12.4f} {tfr_test_results[1]:<12.4f}")
print("=" * 55)

speedup = csv_train_time / tfr_train_time
print(f"\nTraining speedup: TFRecord is {speedup:.1f}x {'faster' if speedup > 1 else 'slower'} than CSV")
print(f"Accuracy difference: {abs(csv_test_results[1] - tfr_test_results[1]):.4f} (should be minimal)")

Metric                         CSV          TFRecord    
Total training time (s)        58.3         51.6        
Time per epoch (s)             11.7         10.3        
Test accuracy                  0.9441       0.9441      

Training speedup: TFRecord is 1.1x faster than CSV
Accuracy difference: 0.0000 (should be minimal)


---
## Section 7: Cleanup & Exam Takeaways

In [ ]:
# Cleanup local files
import shutil

for d in [DATA_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Deleted: {d}")

print("\nCleanup complete.")

### Exam Decision Framework: When to Use What?

| Format | Best For | Avoid When |
|--------|----------|------------|
| **CSV** | Small datasets (<100MB), quick prototyping, human-readable debugging | Large-scale training, distributed training |
| **TFRecord** | Large datasets, TensorFlow training, distributed/multi-GPU, need streaming reads | Non-TF frameworks (PyTorch, sklearn), small datasets |
| **TFRecord + Sharding** | Distributed training, very large datasets, when shuffle quality matters | Single-machine small-data training |
| **Parquet/Avro** | Multi-framework pipelines, columnar queries, Spark/Beam preprocessing | Direct TF training (need conversion) |
| **BigQuery** | Exploration, SQL-based feature engineering, BQML training | Direct custom model training (export first) |

### Key Exam Patterns

1. **"Training is slow on a large dataset"** → Convert to TFRecord + shard + tf.data pipeline with prefetch
2. **"How to improve data loading for distributed training?"** → Sharded TFRecords with interleave
3. **"How to prevent train-serve skew in data preprocessing?"** → Use tf.Transform to generate a transform graph that produces TFRecords during training AND applies the same transforms at serving time
4. **"What file format for TensorFlow training?"** → TFRecord (the default answer unless there's a reason not to)
5. **"Schema management for TFRecords?"** → Feature description must be maintained separately; consider tf.Transform or schema files

### What We Learned

- **TFRecord is a binary serialization format** using protocol buffers, not a compression format — files may be similar size or larger than CSV
- **The speed gain comes from read/parse efficiency**, not file size reduction
- **Feature description (schema) is mandatory at read time** — this is the biggest TFRecord gotcha
- **Sharding enables parallelism** — use interleave() for concurrent reads across shards
- **10-100 shards is typical** — aim for 100-200MB per shard, at least as many shards as workers
- **tf.data.AUTOTUNE** lets TensorFlow optimize parallelism automatically
- **prefetch(AUTOTUNE)** overlaps I/O with compute — always include it